# Acceleration RMS — Jan–Jun 2026 (30 min)

Interactive **HTML** timeline for `AHU_2_9_Blower_DE_A_jan2_jun_30_min.csv`.

- Zoom: scroll wheel or box-select (drag)
- Pan: drag on chart
- Range slider at bottom for quick time window

Output: `AHU_2_9_Blower_DE_A_jan2_jun_30_min_rms_timeline.html` (same folder).

In [ ]:
from __future__ import annotations

from pathlib import Path

import pandas as pd
import plotly.graph_objects as go

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "data").is_dir() and (REPO_ROOT.parent / "data").is_dir():
    REPO_ROOT = REPO_ROOT.parent

DATA_DIR = REPO_ROOT / "data" / "jan2jun_by_sensor"
INPUT_CSV = DATA_DIR / "AHU_2_9_Blower_DE_A_jan2_jun_30_min.csv"
OUTPUT_HTML = DATA_DIR / "AHU_2_9_Blower_DE_A_jan2_jun_30_min_rms_timeline.html"

TIME_COL = "TIMESTAMP"
VALUE_COL = "Acceleration RMS"
SENSOR_NAME = "AHU 2-9 Blower DE A"

print("Input:", INPUT_CSV)
print("Output:", OUTPUT_HTML)

In [ ]:
def parse_timestamp_series(series: pd.Series) -> pd.Series:
    raw = series.astype(str).str.strip()
    parsed = pd.to_datetime(raw, dayfirst=True, format="mixed", errors="coerce")
    for fmt in ("%Y-%m-%d %H:%M:%S", "%Y-%m-%d %H:%M"):
        mask = parsed.isna()
        if not mask.any():
            break
        parsed.loc[mask] = pd.to_datetime(raw.loc[mask], format=fmt, errors="coerce")
    if int(parsed.isna().sum()):
        raise ValueError(f"Failed to parse {int(parsed.isna().sum())} timestamps.")
    return parsed


def load_rms_series(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, low_memory=False)
    if VALUE_COL not in df.columns and "DATA12" in df.columns:
        df[VALUE_COL] = pd.to_numeric(df["DATA12"], errors="coerce")
    if VALUE_COL not in df.columns:
        raise ValueError(f"{path.name}: need {VALUE_COL!r} or DATA12")
    df = df.copy()
    df[TIME_COL] = parse_timestamp_series(df[TIME_COL])
    if "SENSOR_DESC" in df.columns:
        df["SENSOR_DESC"] = df["SENSOR_DESC"].astype(str).str.strip()
        df = df[df["SENSOR_DESC"] == SENSOR_NAME]
    elif "SENSOR_NAME" in df.columns:
        df = df[df["SENSOR_NAME"].astype(str).str.strip() == SENSOR_NAME]
    df[VALUE_COL] = pd.to_numeric(df[VALUE_COL], errors="coerce")
    df = df.dropna(subset=[TIME_COL, VALUE_COL])
    df = df.sort_values(TIME_COL, kind="mergesort").drop_duplicates(TIME_COL, keep="last")
    return df.reset_index(drop=True)


df = load_rms_series(INPUT_CSV)
rms = df[VALUE_COL]
print(f"Rows: {len(df):,}")
print(f"Time: {df[TIME_COL].min()} -> {df[TIME_COL].max()}")
print(f"RMS  min/median/max: {rms.min():.3f} / {rms.median():.3f} / {rms.max():.3f}")
df.head()

In [ ]:
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=df[TIME_COL],
        y=df[VALUE_COL],
        mode="lines",
        name=VALUE_COL,
        line=dict(width=1.2, color="#2563eb"),
        hovertemplate="%{x|%Y-%m-%d %H:%M}<br>RMS=%{y:.3f}<extra></extra>",
    )
)
fig.update_layout(
    title=f"{SENSOR_NAME} — Jan–Jun 2026 (30 min, n={len(df):,})",
    xaxis_title="Time",
    yaxis_title=VALUE_COL,
    template="plotly_white",
    hovermode="x unified",
    height=560,
    margin=dict(l=60, r=30, t=60, b=50),
)
fig.update_xaxes(rangeslider=dict(visible=True))
fig.write_html(str(OUTPUT_HTML), include_plotlyjs="cdn", full_html=True)
print("Wrote:", OUTPUT_HTML)
fig.show()